In [21]:
import pandas as pd
import geopandas as gpd
import numpy as np
import re
from variables_functions import schools, kindergartens, parking_spaces, parks_catchment_areas

neighborhoods_standardized = gpd.read_parquet(r'D:\Data Analysis\5_Projects\sofia_housing_districts_analysis\results\intermediate\neighborhoods_standardized.parquet')
district_codes_corrected_latin = gpd.read_parquet(r'D:\Data Analysis\5_Projects\sofia_housing_districts_analysis\results\intermediate\district_codes_corrected_latin.parquet')

In [22]:
if schools.crs != district_codes_corrected_latin.crs:
    schools = schools.to_crs(district_codes_corrected_latin.crs)

if kindergartens.crs != district_codes_corrected_latin.crs:
    kindergartens = kindergartens.to_crs(district_codes_corrected_latin.crs)

if parking_spaces.crs != district_codes_corrected_latin.crs:
    parking_spaces = parking_spaces.to_crs(district_codes_corrected_latin.crs)


parks_catchment_areas_800m = parks_catchment_areas[parks_catchment_areas['tobreak'] == 800]

parks_catchment_areas_2000m = parks_catchment_areas[parks_catchment_areas['tobreak'] == 2000]

district_codes_corrected_latin = district_codes_corrected_latin.reset_index(drop=True)
district_codes_corrected_latin['district_id'] = district_codes_corrected_latin.index

joined_schools = gpd.sjoin(
    schools,
    district_codes_corrected_latin[['district_id', 'geometry']],
    how='left',
    predicate='within'
)

joined_kindergartens = gpd.sjoin(
    kindergartens,
    district_codes_corrected_latin[['district_id', 'geometry']],
    how='left',
    predicate='within'
)

joined_parking_spaces = gpd.sjoin(
    parking_spaces,
    district_codes_corrected_latin[['district_id', 'geometry']],
    how='left',
    predicate='within'
)


counts_schools = (
    joined_schools.dropna(subset=['district_id'])
    .groupby('district_id')
    .size()
    .reset_index(name='school_count')
)

counts_kindergartens = (
    joined_kindergartens.dropna(subset=['district_id'])
    .groupby('district_id')
    .size()
    .reset_index(name='kindergarten_count')
)

counts_parking_spaces = (
    joined_parking_spaces.dropna(subset=['district_id'])
    .groupby('district_id')
    .size()
    .reset_index(name='parking_space_count')
)



counts_districts = district_codes_corrected_latin.merge(counts_schools, on='district_id', how='left')
counts_districts['school_count'] = counts_districts['school_count'].fillna(0).astype(int)

counts_districts = counts_districts.merge(counts_kindergartens, on='district_id', how='left')
counts_districts['kindergarten_count'] = counts_districts['kindergarten_count'].fillna(0).astype(int)

counts_districts = counts_districts.merge(counts_parking_spaces, on='district_id', how='left')
counts_districts['parking_space_count'] = counts_districts['parking_space_count'].fillna(0).astype(int)


counts_districts.explore()

In [29]:
zones_2000 = parks_catchment_areas_2000m.to_crs(32635).dissolve()
zones_800 = parks_catchment_areas_800m.to_crs(32635).dissolve()
d = district_codes_corrected_latin.to_crs(32635)

covered_2000 = gpd.overlay(d[['obns_lat', 'geometry']], zones_2000[['geometry']], how='intersection')
covered_2000['covered_area_2000'] = covered_2000.geometry.area

covered_800 = gpd.overlay(d[['obns_lat', 'geometry']], zones_800[['geometry']], how='intersection')
covered_800['covered_area_800'] = covered_800.geometry.area

d['area'] = d.geometry.area

share_2000 = d[['obns_lat', 'area', 'geometry']].merge(
    covered_2000.groupby('obns_lat')['covered_area_2000'].sum().reset_index(),
    on='obns_lat', how='left'
)

share_800 = d[['obns_lat', 'area', 'geometry']].merge(
    covered_800.groupby('obns_lat')['covered_area_800'].sum().reset_index(),
    on='obns_lat', how='left'
)


share_2000['park_2000m_pct'] = (share_2000['covered_area_2000'].fillna(0) / share_2000['area'] * 100).round(1)
share_800['park_800m_pct'] = (share_800['covered_area_800'].fillna(0) / share_800['area'] * 100).round(1)


share = share_800.merge(
    share_2000[['obns_lat', 'covered_area_2000', 'park_2000m_pct']],
    on='obns_lat', how='left', validate='1:1'
)

share.explore()


In [24]:
if schools.crs != neighborhoods_standardized.crs:
    schools = schools.to_crs(neighborhoods_standardized.crs)

if kindergartens.crs != neighborhoods_standardized.crs:
    kindergartens = kindergartens.to_crs(neighborhoods_standardized.crs)

if parking_spaces.crs != neighborhoods_standardized.crs:
    parking_spaces = parking_spaces.to_crs(neighborhoods_standardized.crs)


neighborhoods_standardized = neighborhoods_standardized.reset_index()
neighborhoods_standardized['neighborhood_id'] = neighborhoods_standardized.index

joined_schools = gpd.sjoin(
    schools,
    neighborhoods_standardized[['neighborhood_id', 'geometry']],
    how='left',
    predicate='within'
)

joined_kindergartens = gpd.sjoin(
    kindergartens,
    neighborhoods_standardized[['neighborhood_id', 'geometry']],
    how='left',
    predicate='within'
)

joined_parking_spaces = gpd.sjoin(
    parking_spaces,
    neighborhoods_standardized[['neighborhood_id', 'geometry']],
    how='left',
    predicate='within'
)



counts_schools = (
    joined_schools.dropna(subset=['neighborhood_id'])
    .groupby('neighborhood_id')
    .size()
    .reset_index(name='school_count')
)

counts_kindergartens = (
    joined_kindergartens.dropna(subset=['neighborhood_id'])
    .groupby('neighborhood_id')
    .size()
    .reset_index(name='kindergarten_count')
)

counts_parking_spaces = (
    joined_parking_spaces.dropna(subset=['neighborhood_id'])
    .groupby('neighborhood_id')
    .size()
    .reset_index(name='parking_space_count')
)


counts_neighborhoods = neighborhoods_standardized.merge(counts_schools, on='neighborhood_id', how='left')
counts_neighborhoods['school_count'] = counts_neighborhoods['school_count'].fillna(0).astype(int)

counts_neighborhoods = counts_neighborhoods.merge(counts_kindergartens, on='neighborhood_id', how='left')
counts_neighborhoods['kindergarten_count'] = counts_neighborhoods['kindergarten_count'].fillna(0).astype(int)

counts_neighborhoods = counts_neighborhoods.merge(counts_parking_spaces, on='neighborhood_id', how='left')
counts_neighborhoods['parking_space_count'] = counts_neighborhoods['parking_space_count'].fillna(0).astype(int)

counts_neighborhoods.explore()

In [25]:
parks_catchment_areas[parks_catchment_areas['tobreak'] == 800]

,object_id,objectid,facilityid,name_,frombreak,tobreak,id,area_kv_m,geometry
347,939,939.0,235.0,Location 235 : 0 - 800,0.0,800.0,af983071-9e8a-4092-ad83-1a8311ac0c1e,1295145.46,"MULTIPOLYGON (((23.30041 42.7114, 23.30041 42...."
349,946,946.0,237.0,Location 237 : 0 - 800,0.0,800.0,8b254b1f-3fab-4eb4-9655-2c466d096a25,1271169.18,"MULTIPOLYGON (((23.30007 42.7102, 23.30011 42...."
350,947,947.0,233.0,Location 233 : 0 - 800,0.0,800.0,8a1253b4-21f7-4681-b225-35fc1868ff44,1237939.79,"MULTIPOLYGON (((23.29772 42.71147, 23.29773 42..."
351,959,959.0,230.0,Location 230 : 0 - 800,0.0,800.0,4517f807-dfa9-45b3-bd10-a51c174bf593,1269214.72,"MULTIPOLYGON (((23.29792 42.71184, 23.29794 42..."
352,955,955.0,231.0,Location 231 : 0 - 800,0.0,800.0,c6cee5ac-355e-4483-a11b-1cb2c1038fbc,1187624.76,"MULTIPOLYGON (((23.29683 42.71017, 23.29684 42..."
...,...,...,...,...,...,...,...,...,...
1486,1399,1399.0,120.0,Location 120 : 0 - 800,0.0,800.0,bd039960-10eb-4a0d-8e2a-1029f5012ad7,1010570.24,"MULTIPOLYGON (((23.30405 42.6662, 23.3041 42.6..."
1488,1401,1401.0,117.0,Location 117 : 0 - 800,0.0,800.0,a36b08fd-ff1f-48fa-bad1-93e758646959,1169779.05,"MULTIPOLYGON (((23.31436 42.66587, 23.31459 42..."
1489,1403,1403.0,119.0,Location 119 : 0 - 800,0.0,800.0,4bddc87d-bdce-445e-8b52-5878fc3f98db,1192652.26,"MULTIPOLYGON (((23.3138 42.66473, 23.31418 42...."
1491,1405,1405.0,118.0,Location 118 : 0 - 800,0.0,800.0,dc15b8b6-6cbe-41ee-a1e1-ea170f625b00,1205861.65,"MULTIPOLYGON (((23.31394 42.6684, 23.31403 42...."
